# Vector Space Embeddings & Alignment

This notebook covers:
1. **Static Token Architectures**: Word2Vec (Skip-gram & CBOW), GloVe, FastText
2. **Contextual Tokenizers**: Custom token regression heads
3. **Contrastive Architectures**: Dual-Encoder CLIP-style alignment

These embeddings map words/images to a shared vector space.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter
import numpy as np

# =====================================================================
# WORD2VEC: SKIP-GRAM AND CBOW
# =====================================================================

class Word2VecSkipGram(nn.Module):
    """
    Word2Vec Skip-gram model.
    Predicts context words given a center word.
    
    Args:
        vocab_size: Size of vocabulary
        embedding_dim: Dimension of word embeddings
    """
    
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.embedding_dim = embedding_dim
        
        # Center word embedding
        self.center_embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # Context word embedding
        self.context_embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # Initialize embeddings
        nn.init.normal_(self.center_embedding.weight, std=0.02)
        nn.init.normal_(self.context_embedding.weight, std=0.02)
    
    def forward(self, center_words, context_words):
        """
        Args:
            center_words: [batch_size]
            context_words: [batch_size]
        
        Returns:
            logits: [batch_size]
        """
        center_embed = self.center_embedding(center_words)  # [batch, embed_dim]
        context_embed = self.context_embedding(context_words)  # [batch, embed_dim]
        
        # Dot product
        logits = torch.sum(center_embed * context_embed, dim=1)  # [batch]
        
        return logits


class Word2VecCBOW(nn.Module):
    """
    Word2Vec Continuous Bag of Words (CBOW) model.
    Predicts a center word given context words.
    
    Args:
        vocab_size: Size of vocabulary
        embedding_dim: Dimension of word embeddings
        context_size: Number of context words on each side
    """
    
    def __init__(self, vocab_size, embedding_dim, context_size=2):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.context_size = context_size
        
        # Word embedding
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # Projection layer
        self.projection = nn.Linear(embedding_dim, vocab_size)
        
        nn.init.normal_(self.embedding.weight, std=0.02)
    
    def forward(self, context_words):
        """
        Args:
            context_words: [batch_size, context_size*2]
        
        Returns:
            logits: [batch_size, vocab_size]
        """
        embeds = self.embedding(context_words)  # [batch, context_size*2, embed_dim]
        
        # Average context embeddings
        context_embed = embeds.mean(dim=1)  # [batch, embed_dim]
        
        logits = self.projection(context_embed)  # [batch, vocab_size]
        
        return logits


In [ ]:
# =====================================================================
# GLOVE: GLOBAL VECTORS FOR WORD REPRESENTATION
# =====================================================================

class GloVe(nn.Module):
    """
    GloVe: Global Vectors for Word Representation
    Combines the advantages of matrix factorization and local context window methods.
    
    Loss: L = Σ f(X_ij) * (w_i^T * w_j + b_i + b_j - log(X_ij))^2
    
    Args:
        vocab_size: Size of vocabulary
        embedding_dim: Dimension of word embeddings
    """
    
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.embedding_dim = embedding_dim
        
        # Word embeddings
        self.word_embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # Context embeddings
        self.context_embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # Biases
        self.word_bias = nn.Embedding(vocab_size, 1)
        self.context_bias = nn.Embedding(vocab_size, 1)
        
        nn.init.normal_(self.word_embedding.weight, std=0.02)
        nn.init.normal_(self.context_embedding.weight, std=0.02)
        nn.init.zeros_(self.word_bias.weight)
        nn.init.zeros_(self.context_bias.weight)
    
    def forward(self, word_idx, context_idx, cooccurrence_count):
        """
        Args:
            word_idx: [batch_size]
            context_idx: [batch_size]
            cooccurrence_count: [batch_size] Co-occurrence counts X_ij
        
        Returns:
            loss: Scalar loss
        """
        word_embed = self.word_embedding(word_idx)  # [batch, embed_dim]
        context_embed = self.context_embedding(context_idx)  # [batch, embed_dim]
        
        word_bias = self.word_bias(word_idx).squeeze(-1)  # [batch]
        context_bias = self.context_bias(context_idx).squeeze(-1)  # [batch]
        
        # Compute prediction
        prediction = torch.sum(word_embed * context_embed, dim=1) + word_bias + context_bias
        
        # GloVe loss with weighting function f(x) = min(1, (x/x_max)^0.75)
        x_max = 100.0
        weight = torch.clamp((cooccurrence_count / x_max) ** 0.75, max=1.0)
        
        log_cooccurrence = torch.log(cooccurrence_count + 1e-8)
        loss = weight * (prediction - log_cooccurrence) ** 2
        
        return loss.mean()


# =====================================================================
# FASTTEXT: SUBWORD EMBEDDINGS
# =====================================================================

class FastText(nn.Module):
    """
    FastText: Word embeddings with subword information.
    Represents each word as sum of character n-grams.
    
    Args:
        vocab_size: Size of vocabulary (words)
        n_gram_vocab: Size of n-gram vocabulary
        embedding_dim: Dimension of embeddings
    """
    
    def __init__(self, vocab_size, n_gram_vocab, embedding_dim):
        super().__init__()
        # Word embeddings
        self.word_embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # N-gram embeddings
        self.ngram_embedding = nn.Embedding(n_gram_vocab, embedding_dim)
        
        # Output projection
        self.output_embed = nn.Embedding(vocab_size, embedding_dim)
        
        nn.init.normal_(self.word_embedding.weight, std=0.02)
        nn.init.normal_(self.ngram_embedding.weight, std=0.02)
        nn.init.normal_(self.output_embed.weight, std=0.02)
    
    def forward(self, word_idx, ngram_indices):
        """
        Args:
            word_idx: [batch_size] Word indices
            ngram_indices: [batch_size, n_grams_per_word] N-gram indices
        
        Returns:
            word_embedding: [batch_size, embed_dim]
        """
        # Get word embedding
        word_embed = self.word_embedding(word_idx)  # [batch, embed_dim]
        
        # Get n-gram embeddings
        ngram_embeds = self.ngram_embedding(ngram_indices)  # [batch, n_grams, embed_dim]
        
        # Sum n-gram embeddings
        ngram_sum = ngram_embeds.mean(dim=1)  # [batch, embed_dim]
        
        # Combine word and n-gram embeddings
        final_embed = word_embed + ngram_sum
        
        return final_embed


In [ ]:
# =====================================================================
# CLIP-STYLE DUAL ENCODER CONTRASTIVE LEARNING
# =====================================================================

class TextEncoder(nn.Module):
    """Simple LSTM-based text encoder"""
    
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)
    
    def forward(self, text_ids):
        """
        Args:
            text_ids: [batch_size, seq_len]
        
        Returns:
            embedding: [batch_size, output_dim]
        """
        embeds = self.embedding(text_ids)  # [batch, seq_len, embed_dim]
        _, (h_n, _) = self.lstm(embeds)  # h_n: [2, batch, hidden]
        h_n = h_n.transpose(0, 1)  # [batch, 2, hidden]
        h_n = h_n.reshape(h_n.shape[0], -1)  # [batch, hidden*2]
        
        embedding = self.fc(h_n)  # [batch, output_dim]
        embedding = F.normalize(embedding, p=2, dim=1)  # L2 normalize
        
        return embedding


class VisionEncoder(nn.Module):
    """Simple CNN-based vision encoder"""
    
    def __init__(self, output_dim):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.fc = nn.Linear(256, output_dim)
    
    def forward(self, images):
        """
        Args:
            images: [batch_size, 3, height, width]
        
        Returns:
            embedding: [batch_size, output_dim]
        """
        x = self.cnn(images)  # [batch, 256, 1, 1]
        x = x.view(x.size(0), -1)  # [batch, 256]
        embedding = self.fc(x)  # [batch, output_dim]
        embedding = F.normalize(embedding, p=2, dim=1)  # L2 normalize
        
        return embedding


class CLIPModel(nn.Module):
    """
    CLIP-style model for image-text alignment.
    Learns a shared embedding space where similar image-text pairs are close.
    
    Loss: InfoNCE (Contrastive Loss)
    """
    
    def __init__(self, vocab_size, text_embed_dim, text_hidden_dim, 
                 shared_embed_dim, temperature=0.07):
        super().__init__()
        self.text_encoder = TextEncoder(vocab_size, text_embed_dim, 
                                        text_hidden_dim, shared_embed_dim)
        self.vision_encoder = VisionEncoder(shared_embed_dim)
        self.temperature = temperature
    
    def forward(self, image_batch, text_batch):
        """
        Args:
            image_batch: [batch_size, 3, height, width]
            text_batch: [batch_size, seq_len]
        
        Returns:
            image_embeds: [batch_size, shared_embed_dim]
            text_embeds: [batch_size, shared_embed_dim]
        """
        image_embeds = self.vision_encoder(image_batch)
        text_embeds = self.text_encoder(text_batch)
        return image_embeds, text_embeds
    
    def contrastive_loss(self, image_embeds, text_embeds):
        """
        InfoNCE contrastive loss.
        Pulls together matching image-text pairs while pushing apart non-matching pairs.
        
        Args:
            image_embeds: [batch_size, embed_dim]
            text_embeds: [batch_size, embed_dim]
        
        Returns:
            loss: Scalar loss
        """
        # Compute similarity matrix
        logits = torch.matmul(image_embeds, text_embeds.t()) / self.temperature  # [batch, batch]
        
        # Labels are on the diagonal (i-th image matches i-th text)
        labels = torch.arange(len(image_embeds), device=image_embeds.device)
        
        # Cross-entropy loss
        loss_image = F.cross_entropy(logits, labels)
        loss_text = F.cross_entropy(logits.t(), labels)
        
        return (loss_image + loss_text) / 2


In [ ]:
# =====================================================================
# CONTEXTUAL TOKEN REGRESSION
# =====================================================================

class ContextualTokenRegressor(nn.Module):
    """
    Custom token regression head for contextual embeddings.
    Maps contextual token embeddings to continuous token representations.
    """
    
    def __init__(self, hidden_dim, output_dim):
        super().__init__()
        self.fc1 = nn.Linear(hidden_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        self.layer_norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(0.1)
    
    def forward(self, contextual_embeddings):
        """
        Args:
            contextual_embeddings: [batch_size, seq_len, hidden_dim]
        
        Returns:
            token_embeddings: [batch_size, seq_len, output_dim]
        """
        x = self.fc1(contextual_embeddings)
        x = F.gelu(x)
        x = self.layer_norm(x)
        x = self.dropout(x)
        token_embeddings = self.fc2(x)
        
        # L2 normalize
        token_embeddings = F.normalize(token_embeddings, p=2, dim=-1)
        
        return token_embeddings


In [ ]:
# =====================================================================
# DEMONSTRATION AND TESTING
# =====================================================================

if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}\n")
    
    # Test 1: Word2Vec
    print("=" * 60)
    print("WORD2VEC: SKIP-GRAM AND CBOW")
    print("=" * 60)
    
    vocab_size = 10000
    embed_dim = 300
    
    skipgram = Word2VecSkipGram(vocab_size, embed_dim).to(device)
    cbow = Word2VecCBOW(vocab_size, embed_dim, context_size=2).to(device)
    
    center = torch.randint(0, vocab_size, (32,)).to(device)
    context = torch.randint(0, vocab_size, (32,)).to(device)
    context_batch = torch.randint(0, vocab_size, (32, 4)).to(device)
    
    skipgram.eval()
    cbow.eval()
    with torch.no_grad():
        sg_logits = skipgram(center, context)
        cbow_logits = cbow(context_batch)
    
    print(f"Skip-gram output shape: {sg_logits.shape}")
    print(f"CBOW output shape: {cbow_logits.shape}")
    print(f"✓ Word2Vec models working correctly!\n")
    
    # Test 2: CLIP
    print("=" * 60)
    print("CLIP: IMAGE-TEXT CONTRASTIVE ALIGNMENT")
    print("=" * 60)
    
    clip = CLIPModel(vocab_size=10000, text_embed_dim=256, text_hidden_dim=512,
                     shared_embed_dim=512).to(device)
    
    images = torch.randn(16, 3, 224, 224).to(device)
    texts = torch.randint(0, 10000, (16, 77)).to(device)
    
    clip.eval()
    with torch.no_grad():
        img_embeds, txt_embeds = clip(images, texts)
        loss = clip.contrastive_loss(img_embeds, txt_embeds)
    
    print(f"Image embeddings shape: {img_embeds.shape}")
    print(f"Text embeddings shape: {txt_embeds.shape}")
    print(f"Contrastive loss: {loss.item():.4f}")
    print(f"✓ CLIP model working correctly!\n")
    
    # Test 3: Contextual Token Regression
    print("=" * 60)
    print("CONTEXTUAL TOKEN REGRESSION")
    print("=" * 60)
    
    regressor = ContextualTokenRegressor(hidden_dim=768, output_dim=512).to(device)
    
    contextual = torch.randn(8, 77, 768).to(device)
    
    regressor.eval()
    with torch.no_grad():
        token_embeds = regressor(contextual)
    
    print(f"Contextual embeddings input shape: {contextual.shape}")
    print(f"Token embeddings output shape: {token_embeds.shape}")
    print(f"✓ Contextual regression working correctly!\n")
    
    print("✅ All embedding models working!")
